In [1]:
import pandas as pd
import pickle
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight


# ================================
# Path setup
# ================================
def find_repo_root(start_dir: Path) -> Path:
    """Find project root by locating the cleaned dataset path."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "data" / "cleaned" / "data_annotation_clean.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/cleaned/data_annotation_clean.csv from the current working directory."
    )


repo_root = find_repo_root(Path.cwd())
DATA_PATH = repo_root / "data" / "cleaned" / "data_annotation_clean.csv"
ARTIFACTS_DIR = repo_root / "classification" / "svm" / "model_artifacts"
TFIDF_PATH = ARTIFACTS_DIR / "tfidf.pkl"
MODEL_PATH = ARTIFACTS_DIR / "svm_model.pkl"

TEXT_COL_CANDIDATES = ["clean_text", "body_training", "body", "text"]
LABEL_COL = "manual_labelling"


# ================================
# Load dataset
# ================================
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing cleaned dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"Missing required label column: {LABEL_COL}")

text_col = next((c for c in TEXT_COL_CANDIDATES if c in df.columns), None)
if text_col is None:
    raise ValueError(
        f"Missing text column. Tried {TEXT_COL_CANDIDATES}. Available columns: {list(df.columns)}"
    )

TEXT_COL = text_col
print(f"Using text column: {TEXT_COL}")
print(f"Loaded dataset: {DATA_PATH}")

# Keep only needed columns and sanitize input
df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()
df = df[(df[TEXT_COL] != "") & (df[LABEL_COL] != "")]

print("Dataset size:", len(df))
print("Class distribution (full dataset):")
print(df[LABEL_COL].value_counts())

# ================================
# Train-test split
# ================================
# Fallback if a class has <2 samples (stratified split would fail).
label_counts = df[LABEL_COL].value_counts()
if (label_counts < 2).any():
    print("\nWarning: Some classes have <2 samples. Falling back to non-stratified split.")
    stratify_labels = None
else:
    stratify_labels = df[LABEL_COL]

X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL],
    df[LABEL_COL],
    test_size=0.2,
    stratify=stratify_labels,
    random_state=42,
)

# ================================
# TF-IDF Vectorizer
# ================================
# Tuned for noisy social text while preserving your baseline setup.
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=False,  # text is already preprocessed in Step 1
    dtype=np.float32,
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# ================================
# Class imbalance handling
# ================================
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {cls: float(w) for cls, w in zip(classes, class_weights)}

print("\nComputed class weights:")
for cls in classes:
    print(f"  {cls}: {class_weight_dict[cls]:.4f}")

# ================================
# SVM Model (with calibration)
# ================================
svm = LinearSVC(class_weight=class_weight_dict, random_state=42, max_iter=5000)

# Keep calibration for reliable confidence estimates in Step 2 inference.
model = CalibratedClassifierCV(svm, method="sigmoid", cv=5)
model.fit(X_train_vec, y_train)

# ================================
# Evaluation (important for report)
# ================================
y_pred = model.predict(X_test_vec)
labels = sorted(y_test.unique())

print("\n=== Evaluation Summary ===")
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy:           {acc:.4f}")
print(f"Balanced Accuracy:  {bal_acc:.4f}")
print(f"Macro F1:           {macro_f1:.4f}")
print(f"Weighted F1:        {weighted_f1:.4f}")

print("\nPer-class Precision / Recall / F1:")
print(classification_report(y_test, y_pred, labels=labels, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels],
)
print("Confusion Matrix (counts; rows=true, cols=pred):")
print(cm_df)

cm_norm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
cm_norm_df = pd.DataFrame(
    np.round(cm_norm, 4),
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels],
)
print("\nConfusion Matrix (row-normalized):")
print(cm_norm_df)

# ================================
# Save model artifacts
# ================================
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
with open(TFIDF_PATH, "wb") as f:
    pickle.dump(tfidf, f)

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

print(f"\nModels saved:\n- {TFIDF_PATH}\n- {MODEL_PATH}")

Using text column: clean_text
Loaded dataset: /Users/dylan/Documents/GitHub/sc4021-ir/data/cleaned/data_annotation_clean.csv
Dataset size: 1019
Class distribution (full dataset):
manual_labelling
Positive    445
Neutral     317
Negative    257
Name: count, dtype: int64

Computed class weights:
  Negative: 1.3188
  Neutral: 1.0738
  Positive: 0.7631

=== Evaluation Summary ===
Accuracy:           0.4951
Balanced Accuracy:  0.4202
Macro F1:           0.3861
Weighted F1:        0.4301

Per-class Precision / Recall / F1:
              precision    recall  f1-score   support

    Negative     0.6667    0.0784    0.1404        51
     Neutral     0.5122    0.3281    0.4000        64
    Positive     0.4841    0.8539    0.6179        89

    accuracy                         0.4951       204
   macro avg     0.5543    0.4202    0.3861       204
weighted avg     0.5385    0.4951    0.4301       204

Confusion Matrix (counts; rows=true, cols=pred):
               pred_Negative  pred_Neutral  pre